# MSBA 6680 Capstone Project
## Portfolio Construction & Asset Pricing Analytics

**Student:** Bryson  
**Stocks:** Coca-Cola (KO), Lululemon (LULU), Nike (NKE), Planet Fitness (PLNT)  
**Benchmark:** S&P 500  
**Data Source:** FactSet + Fama-French Data Library (Kenneth French, Dartmouth)  
**Period:** April 2025 – February 2026 (daily data)

---

This notebook walks through portfolio construction, risk analytics, and asset pricing analysis for a four-stock consumer and lifestyle portfolio. The workflow follows the analytical steps used by asset managers and investment strategists: load clean data, study return and risk behavior, build portfolios, and then decompose returns using factor models.

# 1. Project Overview & Investment Objective

*[TO BE COMPLETED AFTER TUESDAY DISCUSSION]*

### Investment Objective
*[Brief statement of portfolio objective — e.g., "Build a consumer-lifestyle portfolio..."]*

### Economic Motivation
*[Why these four stocks — the story behind the selection]*

### Stocks Selected
- **KO (Coca-Cola):** Consumer staples — defensive, dividend-paying, historically low-beta
- **LULU (Lululemon):** Consumer discretionary — premium athleisure, growth-oriented
- **NKE (Nike):** Consumer discretionary — global brand, cyclical exposure
- **PLNT (Planet Fitness):** Consumer discretionary — fitness/wellness, domestic-focused

### Why This Matters
There is no free lunch in investing — you must take risks to earn a return. The goal of this project is to sort out which portion of returns comes from taking risk (factor exposures) and which portion comes from skill or luck (alpha). A clear framework lets us evaluate whether a portfolio is earning its keep.

# 2. Data Selection & Preparation

We use FactSet for daily price history of four public companies and the S&P 500 benchmark. Fama-French factor returns (MKT-RF, SMB, HML, RMW, CMA) and the risk-free rate come from the Kenneth French Data Library at Dartmouth.

Documentation: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm
from scipy.optimize import minimize

# Basic plot settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Libraries loaded.")

In [ ]:
# Load FactSet price data for the four stocks
# Each file has a header row and units row on top, so skiprows=2

stock_files = {
    'KO':   'Cola.xlsx',
    'LULU': 'lulu.xlsx',
    'NKE':  'NIKE.xlsx',
    'PLNT': 'Planet_Fitnes.xlsx',
}

prices = {}
for ticker, filename in stock_files.items():
    df = pd.read_excel(filename, sheet_name='Price History', skiprows=2)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    prices[ticker] = df['Price']
    print(f"Loaded {ticker}: {len(df)} trading days")

# Load S&P 500 benchmark
sp = pd.read_excel('S_P_500.xlsx', sheet_name='Price History', skiprows=2)
sp['Date'] = pd.to_datetime(sp['Date'])
sp = sp.sort_values('Date').set_index('Date')
prices['SP500'] = sp['Price']
print(f"Loaded SP500: {len(sp)} trading days")

# Merge into one price DataFrame
price_df = pd.DataFrame(prices).dropna()
print(f"\nMerged price data: {price_df.shape[0]} common trading days")
print(f"Date range: {price_df.index.min().date()} to {price_df.index.max().date()}")
price_df.head()

In [ ]:
# Load Fama-French 5-Factor Daily data
# Source: Kenneth French Data Library
# Download the file "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip" and unzip to get FF.csv

# Read the CSV (skip the 4-line header)
df_ff = pd.read_csv('FF.csv', skiprows=4)

# The first column is the date, which needs cleanup
df_ff.columns = df_ff.columns.str.strip()
df_ff = df_ff.rename(columns={df_ff.columns[0]: 'Date'})
df_ff['Date'] = pd.to_datetime(df_ff['Date'].astype(str).str.strip(), 
                                format='%Y%m%d', errors='coerce')
df_ff = df_ff.dropna(subset=['Date']).set_index('Date')

# Convert from percent to decimal (FF data is stored in percentage points)
for col in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']:
    df_ff[col] = pd.to_numeric(df_ff[col], errors='coerce') / 100

# Filter to our date range
ff = df_ff.loc[price_df.index.min():price_df.index.max()]
print(f"Fama-French factors: {ff.shape[0]} trading days")
print(f"Date range: {ff.index.min().date()} to {ff.index.max().date()}")
ff.head()

In [ ]:
# Compute daily simple returns for all stocks
stock_tickers = ['KO', 'LULU', 'NKE', 'PLNT']

returns = price_df[stock_tickers].pct_change().dropna()
sp500_ret = price_df['SP500'].pct_change().dropna()

# Align everything on common dates (FF data may end earlier than price data)
common_dates = returns.index.intersection(ff.index).intersection(sp500_ret.index)
returns = returns.loc[common_dates]
sp500_ret = sp500_ret.loc[common_dates]
ff = ff.loc[common_dates]

# Compute excess returns (stock return minus risk-free rate)
# This is what we'll use as the dependent variable in factor regressions
exret = returns.subtract(ff['RF'], axis=0)

print(f"Analysis period: {common_dates.min().date()} to {common_dates.max().date()}")
print(f"Trading days: {len(common_dates)}")
print(f"\nFirst few daily returns:")
returns.head()

# 3. Financial Time-Series & Volatility Analysis

This section looks at how each stock behaves through time — the shape of returns, how much they move around (volatility), and whether volatility clusters in specific periods.

**Key concepts:**
- **Daily return:** $R_t = (P_t - P_{t-1}) / P_{t-1}$
- **Annualized volatility:** daily standard deviation × √252 (there are about 252 trading days per year)
- **Volatility clustering:** periods of high volatility tend to be followed by more high volatility, and calm periods by more calm periods

In [ ]:
# Summary statistics — daily and annualized

avg_returns = returns.mean()
std_returns = returns.std()

summary = pd.DataFrame({
    "Avg daily return": avg_returns,
    "Daily std dev": std_returns,
    "Annualized return": avg_returns * 252,
    "Annualized vol": std_returns * np.sqrt(252),
    "Min daily return": returns.min(),
    "Max daily return": returns.max(),
    "Skewness": returns.skew(),
    "Kurtosis": returns.kurt(),
})

print("Individual Stock Daily Risk & Return:")
summary.round(4)

In [ ]:
# Normalized price chart — rebase everything to 100 at the start
# This lets us compare performance across stocks of different price levels

normalized = (price_df[stock_tickers + ['SP500']] / 
              price_df[stock_tickers + ['SP500']].iloc[0]) * 100

plt.figure(figsize=(14, 7))
colors = {'KO': '#D12020', 'LULU': '#E8457C', 'NKE': '#F57C00', 
          'PLNT': '#7B1FA2', 'SP500': '#333333'}

for col in normalized.columns:
    linestyle = '--' if col == 'SP500' else '-'
    plt.plot(normalized.index, normalized[col], linestyle=linestyle,
             label=col, color=colors[col], linewidth=1.8)

plt.title('Normalized Price Performance (Base = 100)')
plt.xlabel('Date')
plt.ylabel('Indexed Price')
plt.axhline(100, color='gray', linestyle=':', alpha=0.5)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Price and rolling volatility — stacked view
# Top panel: S&P 500 price
# Bottom panel: 30-day rolling annualized volatility for each stock

plt.figure(figsize=(14, 10))

# Top: S&P 500 price
plt.subplot(2, 1, 1)
plt.plot(price_df.index, price_df['SP500'], color='blue', label='S&P 500')
plt.title('S&P 500 Price')
plt.ylabel('Price')
plt.legend()

# Bottom: rolling volatility of each stock
plt.subplot(2, 1, 2)
rolling_vol = returns.rolling(window=30).std() * np.sqrt(252)
for ticker in stock_tickers:
    plt.plot(rolling_vol.index, rolling_vol[ticker], 
             label=ticker, color=colors[ticker], linewidth=1.5)

plt.title('30-Day Rolling Annualized Volatility')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Return distributions — histograms for each stock
# Shows shape, skewness, and fat-tail behavior

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ticker in enumerate(stock_tickers):
    ax = axes[i]
    ax.hist(returns[ticker], bins=40, alpha=0.7, color=colors[ticker], 
            edgecolor='white', density=True)
    ax.axvline(returns[ticker].mean(), color='black', linestyle='--', 
               label=f'Mean: {returns[ticker].mean()*100:.3f}%')
    ax.set_title(f'{ticker} Daily Return Distribution')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Daily Return Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Volatility clustering — squared returns over time
# Clusters of large squared returns reveal periods of sustained high volatility

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ticker in enumerate(stock_tickers):
    ax = axes[i]
    squared = (returns[ticker] ** 2) * 10000  # scale to basis points squared
    ax.bar(squared.index, squared.values, color=colors[ticker], alpha=0.7, width=1.5)
    ax.set_title(f'{ticker} Squared Returns (Volatility Clustering)')
    ax.set_ylabel('Squared Return (bps²)')

plt.suptitle('Volatility Clustering Evidence', y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation: Clusters of tall bars indicate periods of sustained")
print("high volatility — a well-known feature of financial returns.")

### Time-Series & Volatility Interpretation

*[Fill in after running — key points to address:]*
- Which stock had the highest/lowest volatility over the period?
- Any clear periods of market stress visible in the rolling volatility plot?
- Do the squared return plots show obvious volatility clustering?
- How do the return distributions compare across stocks (fat tails, skewness)?

# 4. Portfolio Construction & Risk Analysis

We construct and compare three portfolios:
1. **Equal-Weighted** — 25% in each stock (naive diversification baseline)
2. **Minimum-Variance** — weights chosen to minimize portfolio volatility
3. **Maximum Sharpe** — weights chosen to maximize risk-adjusted return

For each portfolio we compute return, volatility, correlation structure, and downside risk measures (VaR and CVaR).

In [ ]:
# Correlation matrix for the four stocks
# Diversification benefit is strongest when correlations are low

corr = returns.corr()

# Plot correlation heatmap
plt.figure(figsize=(8, 6))
plt.imshow(corr.values, aspect='auto', cmap='RdYlBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(stock_tickers)), stock_tickers, rotation=25, ha='right')
plt.yticks(range(len(stock_tickers)), stock_tickers)
plt.title('Return Correlation Matrix')

# Annotate each cell with its value
for i in range(len(stock_tickers)):
    for j in range(len(stock_tickers)):
        plt.text(j, i, f"{corr.values[i,j]:.2f}", 
                 ha='center', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print("\nCorrelation matrix:")
print(corr.round(3))

In [ ]:
# Compute annualized expected returns and covariance matrix
# We will feed these into the portfolio optimizers below

mu = returns.mean().values * 252          # annualized expected returns
cov = returns.cov().values * 252          # annualized covariance matrix
n = len(stock_tickers)                    # number of assets
rf_annual = ff['RF'].mean() * 252         # annualized risk-free rate

print("Annualized expected returns:")
for t, r in zip(stock_tickers, mu):
    print(f"  {t}: {r*100:.2f}%")

print(f"\nAnnualized risk-free rate: {rf_annual*100:.2f}%")

print("\nAnnualized covariance matrix:")
print(pd.DataFrame(cov, index=stock_tickers, columns=stock_tickers).round(4))

In [ ]:
# Portfolio performance helper function
# Returns: annualized return, annualized vol, Sharpe ratio

def portfolio_performance(weights, mu, cov, rf=rf_annual):
    port_return = weights @ mu
    port_vol = np.sqrt(weights @ cov @ weights)
    sharpe = (port_return - rf) / port_vol
    return port_return, port_vol, sharpe


# Portfolio 1: Equal-weighted (naive baseline)
ew_weights = np.array([1/n] * n)
ew_ret, ew_vol, ew_sharpe = portfolio_performance(ew_weights, mu, cov)

print("=" * 50)
print("EQUAL-WEIGHTED PORTFOLIO")
print("=" * 50)
for t, w in zip(stock_tickers, ew_weights):
    print(f"  {t}: {w*100:.2f}%")
print(f"\nAnnualized return: {ew_ret*100:.3f}%")
print(f"Annualized volatility: {ew_vol*100:.3f}%")
print(f"Sharpe ratio: {ew_sharpe:.4f}")

In [ ]:
# Portfolio 2: Minimum-Variance Portfolio
# Objective: minimize variance subject to weights summing to 1, long-only

def portfolio_variance(w, cov):
    return w.T @ cov @ w

def constraint_sum_to_one(w):
    return np.sum(w) - 1.0

bounds = [(0.0, 1.0)] * n
constraints = [{'type': 'eq', 'fun': constraint_sum_to_one}]
w0 = np.ones(n) / n  # equal-weight starting point

result = minimize(portfolio_variance, w0, args=(cov,), method='SLSQP',
                  bounds=bounds, constraints=constraints)

if not result.success:
    raise RuntimeError("Min-variance optimization failed: " + result.message)

mv_weights = result.x
mv_ret, mv_vol, mv_sharpe = portfolio_performance(mv_weights, mu, cov)

print("=" * 50)
print("MINIMUM-VARIANCE PORTFOLIO")
print("=" * 50)
for t, w in zip(stock_tickers, mv_weights):
    print(f"  {t}: {w*100:.2f}%")
print(f"\nAnnualized return: {mv_ret*100:.3f}%")
print(f"Annualized volatility: {mv_vol*100:.3f}%")
print(f"Sharpe ratio: {mv_sharpe:.4f}")

In [ ]:
# Portfolio 3: Maximum Sharpe (Tangency) Portfolio
# Objective: maximize Sharpe ratio subject to weights summing to 1, long-only
# scipy minimizes, so we minimize the NEGATIVE Sharpe ratio

def neg_sharpe(w, mu, cov, rf):
    port_ret = w @ mu
    port_vol = np.sqrt(w @ cov @ w)
    return -(port_ret - rf) / port_vol

result = minimize(neg_sharpe, w0, args=(mu, cov, rf_annual), method='SLSQP',
                  bounds=bounds, constraints=constraints)

if not result.success:
    raise RuntimeError("Max-Sharpe optimization failed: " + result.message)

ms_weights = result.x
ms_ret, ms_vol, ms_sharpe = portfolio_performance(ms_weights, mu, cov)

print("=" * 50)
print("MAXIMUM SHARPE PORTFOLIO")
print("=" * 50)
for t, w in zip(stock_tickers, ms_weights):
    print(f"  {t}: {w*100:.2f}%")
print(f"\nAnnualized return: {ms_ret*100:.3f}%")
print(f"Annualized volatility: {ms_vol*100:.3f}%")
print(f"Sharpe ratio: {ms_sharpe:.4f}")

In [ ]:
# Investment opportunity set — Monte Carlo simulation
# Generate thousands of random portfolios to visualize the efficient frontier

n_portfolios = 10000
mc_results = np.zeros((3, n_portfolios))

np.random.seed(42)
for i in range(n_portfolios):
    # Random weights from a Dirichlet distribution (sum to 1, all positive)
    w = np.random.dirichlet(np.ones(n))
    ret, vol, sh = portfolio_performance(w, mu, cov)
    mc_results[0, i] = ret
    mc_results[1, i] = vol
    mc_results[2, i] = sh

# Plot the opportunity set + our three portfolios
plt.figure(figsize=(12, 8))
plt.scatter(mc_results[1] * 100, mc_results[0] * 100, c=mc_results[2],
            cmap='viridis', alpha=0.3, s=10)
plt.colorbar(label='Sharpe Ratio')

# Mark the three optimized portfolios
plt.scatter(ew_vol*100, ew_ret*100, marker='D', color='blue', s=220,
            edgecolors='white', linewidth=2, 
            label=f'Equal-Weight (Sharpe: {ew_sharpe:.2f})', zorder=5)
plt.scatter(mv_vol*100, mv_ret*100, marker='*', color='red', s=300,
            edgecolors='white', linewidth=2,
            label=f'Min-Variance (Sharpe: {mv_sharpe:.2f})', zorder=5)
plt.scatter(ms_vol*100, ms_ret*100, marker='P', color='gold', s=260,
            edgecolors='black', linewidth=2,
            label=f'Max Sharpe (Sharpe: {ms_sharpe:.2f})', zorder=5)

plt.title('Investment Opportunity Set with Optimal Portfolios')
plt.xlabel('Annualized Volatility (%)')
plt.ylabel('Annualized Return (%)')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Portfolio weight comparison

weight_df = pd.DataFrame({
    'Equal-Weight': ew_weights,
    'Min-Variance': mv_weights,
    'Max Sharpe': ms_weights
}, index=stock_tickers)

fig, ax = plt.subplots(figsize=(10, 6))
weight_df.plot(kind='bar', ax=ax, 
               color=['#2196F3', '#F44336', '#FFD700'], 
               edgecolor='white', width=0.7)
ax.set_title('Portfolio Weight Comparison')
ax.set_ylabel('Weight')
ax.set_xticklabels(stock_tickers, rotation=0)
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("Weight Summary:")
print(weight_df.round(4))

In [ ]:
# VaR and CVaR at the 95% and 99% confidence levels
# Both historical and parametric (normal) methods
#
# VaR = "I'm X% sure I won't lose more than this in one day"
# CVaR = "When things DO go wrong beyond VaR, here's the average loss"

# Compute daily portfolio returns for each of the three portfolios
port_rets = {
    'Equal-Weight': (returns * ew_weights).sum(axis=1),
    'Min-Variance': (returns * mv_weights).sum(axis=1),
    'Max Sharpe': (returns * ms_weights).sum(axis=1),
}

print("=" * 75)
print("VALUE AT RISK & CONDITIONAL VAR")
print("=" * 75)

for conf_level in [0.95, 0.99]:
    alpha = 1 - conf_level
    print(f"\n--- {conf_level:.0%} Confidence Level ---")
    print(f"{'Portfolio':<15} {'Hist VaR':>12} {'Hist CVaR':>12} {'Norm VaR':>12} {'Norm CVaR':>12}")
    print("-" * 75)
    
    for name, rets in port_rets.items():
        # Historical method: just take the empirical percentile
        var_hist = np.percentile(rets, alpha * 100)
        cvar_hist = rets[rets <= var_hist].mean()
        
        # Parametric normal method: assume returns are normally distributed
        mu_r = rets.mean()
        sigma_r = rets.std(ddof=1)
        z = norm.ppf(alpha)
        var_norm = mu_r + z * sigma_r
        cvar_norm = mu_r - sigma_r * norm.pdf(z) / alpha
        
        print(f"{name:<15} {var_hist*100:>11.3f}% {cvar_hist*100:>11.3f}% "
              f"{var_norm*100:>11.3f}% {cvar_norm*100:>11.3f}%")

print("\nInterpretation:")
print("- Historical VaR captures actual tail behavior (good for fat tails)")
print("- Normal VaR assumes returns are normally distributed (often underestimates")
print("  real-world tail risk because financial returns have fatter tails)")

In [ ]:
# Mini backtest: VaR exceptions
# For a 99% 1-day VaR, we expect about 1% of days to fall below the threshold
# Too many exceptions means VaR is understating risk; too few means it's overstating

print("=" * 75)
print("VaR EXCEPTIONS BACKTEST (99% Confidence)")
print("=" * 75)
print(f"Expected exception rate: 1.00%\n")
print(f"{'Portfolio':<15} {'Hist Excep':>15} {'Norm Excep':>15} {'Hist Rate':>12} {'Norm Rate':>12}")
print("-" * 75)

alpha = 0.01
for name, rets in port_rets.items():
    var_hist = np.percentile(rets, alpha * 100)
    mu_r = rets.mean()
    sigma_r = rets.std(ddof=1)
    var_norm = mu_r + norm.ppf(alpha) * sigma_r
    
    exc_hist = int((rets <= var_hist).sum())
    exc_norm = int((rets <= var_norm).sum())
    total = len(rets)
    
    print(f"{name:<15} {exc_hist:>15} {exc_norm:>15} "
          f"{exc_hist/total*100:>11.2f}% {exc_norm/total*100:>11.2f}%")

print("\nInterpretation: If the actual exception rate is much higher than 1%,")
print("our VaR model is understating downside risk.")

In [ ]:
# Cumulative portfolio returns vs. S&P 500

plt.figure(figsize=(14, 7))
port_colors = {'Equal-Weight': '#2196F3', 'Min-Variance': '#F44336', 'Max Sharpe': '#FFD700'}

for name, rets in port_rets.items():
    cum = (1 + rets).cumprod()
    plt.plot(cum.index, cum, label=name, color=port_colors[name], linewidth=2)

# Benchmark
sp_cum = (1 + sp500_ret).cumprod()
plt.plot(sp_cum.index, sp_cum, '--', label='S&P 500', color='#333333', linewidth=1.5)

plt.title('Cumulative Portfolio Returns vs. S&P 500')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.axhline(1, color='gray', linestyle=':', alpha=0.5)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Portfolio Construction Interpretation

*[Fill in after running — key points to address:]*
- How different are the three portfolios' weight allocations?
- Which had the best risk-adjusted return (Sharpe ratio)?
- What does the correlation matrix tell us about diversification benefit?
- How do historical and parametric VaR compare? Does one understate risk?
- Did any portfolio beat the S&P 500 over the period?

# 5. Asset Pricing Models & Factor Analysis

We decompose individual stock and portfolio returns using three asset pricing models:

1. **CAPM (Capital Asset Pricing Model)** — one factor: market risk
   $$E(R_i) = R_f + \beta_i (R_m - R_f)$$
   
2. **Fama-French 3-Factor Model** — adds size (SMB) and value (HML) factors

3. **Fama-French 5-Factor Model** — adds profitability (RMW) and investment (CMA) factors

**Why this matters:** Most returns come from taking risk — these are not arbitraged away. Factor models help separate returns from taking risk (beta) from returns from skill or luck (alpha). A positive and significant alpha after controlling for factor exposures would suggest genuine skill or a market anomaly. 

We follow the convention that `ret-Rf` is the excess stock return (dependent variable) and `Rm-Rf`, `SMB`, `HML`, `RMW`, `CMA` are the factor returns (independent variables).

In [ ]:
# CAPM regression for each stock
# Y = stock excess return (ret-Rf)
# X = market excess return (Rm-Rf)

print("=" * 75)
print("CAPM REGRESSION RESULTS")
print("=" * 75)

capm_results = {}
for ticker in stock_tickers:
    # Y: excess stock return
    Y = exret[ticker]
    
    # X: market premium (Mkt-RF from Fama-French)
    X = sm.add_constant(ff['Mkt-RF'])
    
    # Run the regression
    model = sm.OLS(Y, X).fit()
    capm_results[ticker] = model
    
    alpha = model.params['const']
    beta = model.params['Mkt-RF']
    alpha_p = model.pvalues['const']
    beta_p = model.pvalues['Mkt-RF']
    
    print(f"\n{ticker}:")
    print(f"  Alpha (daily): {alpha:.6f}   (p-value: {alpha_p:.4f})")
    print(f"  Beta:          {beta:.4f}      (p-value: {beta_p:.4f})")
    print(f"  R-squared:     {model.rsquared:.4f}")

In [ ]:
# Full regression summary for one stock as an example
# (The rest are in capm_results dictionary)

print("DETAILED CAPM REGRESSION — KO")
print("=" * 75)
print(capm_results['KO'].summary())

In [ ]:
# CAPM scatter plot — regression line for each stock

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ticker in enumerate(stock_tickers):
    ax = axes[i]
    model = capm_results[ticker]
    
    x_vals = ff['Mkt-RF']
    y_vals = exret[ticker]
    
    ax.scatter(x_vals * 100, y_vals * 100, alpha=0.5, 
               color=colors[ticker], s=15, label='Daily returns')
    
    # Regression line
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    X_pred = sm.add_constant(x_line)
    y_line = model.predict(X_pred)
    ax.plot(x_line * 100, y_line * 100, color='red', linewidth=2,
            label=f'β = {model.params["Mkt-RF"]:.3f}')
    
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'{ticker} CAPM Regression (R² = {model.rsquared:.3f})')
    ax.set_xlabel('Market Excess Return (Rm-Rf) %')
    ax.set_ylabel(f'{ticker} Excess Return (ret-Rf) %')
    ax.legend()

plt.suptitle('CAPM: Stock Excess Return vs. Market Excess Return', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Fama-French 3-Factor regressions
# Y = stock excess return
# X = Mkt-RF, SMB, HML

print("=" * 75)
print("FAMA-FRENCH 3-FACTOR REGRESSION RESULTS")
print("=" * 75)

ff3_results = {}
for ticker in stock_tickers:
    Y = exret[ticker]
    X = sm.add_constant(ff[['Mkt-RF', 'SMB', 'HML']])
    model = sm.OLS(Y, X).fit()
    ff3_results[ticker] = model
    
    print(f"\n{ticker}:")
    print(f"  Alpha:     {model.params['const']:.6f}  (p={model.pvalues['const']:.4f})")
    print(f"  Mkt-RF:    {model.params['Mkt-RF']:.4f}    (p={model.pvalues['Mkt-RF']:.4f})")
    print(f"  SMB:       {model.params['SMB']:.4f}    (p={model.pvalues['SMB']:.4f})")
    print(f"  HML:       {model.params['HML']:.4f}    (p={model.pvalues['HML']:.4f})")
    print(f"  R-squared: {model.rsquared:.4f}")

In [ ]:
# Full FF3 regression summary for one stock

print("DETAILED FF3 REGRESSION — KO")
print("=" * 75)
print(ff3_results['KO'].summary())

In [ ]:
# Fama-French 5-Factor regressions
# Adds RMW (profitability) and CMA (investment) factors

print("=" * 75)
print("FAMA-FRENCH 5-FACTOR REGRESSION RESULTS")
print("=" * 75)

ff5_results = {}
for ticker in stock_tickers:
    Y = exret[ticker]
    X = sm.add_constant(ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
    model = sm.OLS(Y, X).fit()
    ff5_results[ticker] = model
    
    print(f"\n{ticker}:")
    print(f"  Alpha:     {model.params['const']:.6f}  (p={model.pvalues['const']:.4f})")
    print(f"  Mkt-RF:    {model.params['Mkt-RF']:.4f}    (p={model.pvalues['Mkt-RF']:.4f})")
    print(f"  SMB:       {model.params['SMB']:.4f}    (p={model.pvalues['SMB']:.4f})")
    print(f"  HML:       {model.params['HML']:.4f}    (p={model.pvalues['HML']:.4f})")
    print(f"  RMW:       {model.params['RMW']:.4f}    (p={model.pvalues['RMW']:.4f})")
    print(f"  CMA:       {model.params['CMA']:.4f}    (p={model.pvalues['CMA']:.4f})")
    print(f"  R-squared: {model.rsquared:.4f}")

In [ ]:
# Model comparison across CAPM, FF3, FF5
# We care about: does adding factors explain more return variation?
# R² should increase from CAPM → FF3 → FF5

comparison = pd.DataFrame({
    'CAPM Alpha': [capm_results[t].params['const'] for t in stock_tickers],
    'CAPM Beta': [capm_results[t].params['Mkt-RF'] for t in stock_tickers],
    'CAPM R²': [capm_results[t].rsquared for t in stock_tickers],
    'FF3 Alpha': [ff3_results[t].params['const'] for t in stock_tickers],
    'FF3 R²': [ff3_results[t].rsquared for t in stock_tickers],
    'FF5 Alpha': [ff5_results[t].params['const'] for t in stock_tickers],
    'FF5 R²': [ff5_results[t].rsquared for t in stock_tickers],
}, index=stock_tickers)

print("MODEL COMPARISON: Alpha & R² Across CAPM, FF3, FF5")
print("=" * 75)
print(comparison.round(6))

In [ ]:
# R-squared comparison chart

r2_df = pd.DataFrame({
    'CAPM': [capm_results[t].rsquared for t in stock_tickers],
    'FF3': [ff3_results[t].rsquared for t in stock_tickers],
    'FF5': [ff5_results[t].rsquared for t in stock_tickers],
}, index=stock_tickers)

fig, ax = plt.subplots(figsize=(10, 6))
r2_df.plot(kind='bar', ax=ax, 
           color=['#2196F3', '#F44336', '#4CAF50'], 
           edgecolor='white', width=0.7)
ax.set_title('R-Squared Comparison: CAPM vs FF3 vs FF5')
ax.set_ylabel('R-Squared')
ax.set_xticklabels(stock_tickers, rotation=0)
ax.legend(title='Model')
ax.set_ylim(0, max(r2_df.values.max() * 1.15, 0.5))
plt.tight_layout()
plt.show()

print("\nR-Squared comparison:")
print(r2_df.round(4))

In [ ]:
# Factor loading heatmap — FF5 factors for all stocks
# Positive loading = stock behaves like the factor (e.g., small-cap, value, etc.)
# Negative loading = stock behaves opposite to the factor

factor_names = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
loadings = pd.DataFrame({
    ticker: [ff5_results[ticker].params[f] for f in factor_names]
    for ticker in stock_tickers
}, index=factor_names)

plt.figure(figsize=(9, 6))
plt.imshow(loadings.values, aspect='auto', cmap='RdBu_r',
           vmin=-abs(loadings.values).max(), vmax=abs(loadings.values).max())
plt.colorbar(label='Factor Loading')
plt.xticks(range(len(stock_tickers)), stock_tickers)
plt.yticks(range(len(factor_names)), factor_names)
plt.title('Fama-French 5-Factor Loadings')

for i in range(len(factor_names)):
    for j in range(len(stock_tickers)):
        plt.text(j, i, f"{loadings.values[i,j]:.3f}", 
                 ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Portfolio-level regressions
# Does alpha persist at the portfolio level after factor adjustment?

print("=" * 75)
print("PORTFOLIO-LEVEL CAPM AND FF5 REGRESSIONS")
print("=" * 75)

for name, weights in [('Equal-Weight', ew_weights), 
                      ('Min-Variance', mv_weights), 
                      ('Max Sharpe', ms_weights)]:
    # Compute portfolio's daily excess return
    port_exret = (exret * weights).sum(axis=1)
    
    # CAPM
    X_capm = sm.add_constant(ff['Mkt-RF'])
    capm_m = sm.OLS(port_exret, X_capm).fit()
    
    # FF5
    X_ff5 = sm.add_constant(ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
    ff5_m = sm.OLS(port_exret, X_ff5).fit()
    
    print(f"\n--- {name} Portfolio ---")
    print(f"  CAPM: Alpha={capm_m.params['const']:.6f} (p={capm_m.pvalues['const']:.4f}), "
          f"Beta={capm_m.params['Mkt-RF']:.4f}, R²={capm_m.rsquared:.4f}")
    print(f"  FF5:  Alpha={ff5_m.params['const']:.6f} (p={ff5_m.pvalues['const']:.4f}), "
          f"R²={ff5_m.rsquared:.4f}")
    print(f"        Loadings: Mkt-RF={ff5_m.params['Mkt-RF']:.3f}, "
          f"SMB={ff5_m.params['SMB']:.3f}, HML={ff5_m.params['HML']:.3f}, "
          f"RMW={ff5_m.params['RMW']:.3f}, CMA={ff5_m.params['CMA']:.3f}")

### Asset Pricing Interpretation

*[Fill in after running — key points to address:]*
- Which stocks have the highest and lowest market beta?
- Does any stock show statistically significant alpha (p < 0.05) after factor adjustment?
- How much does R² improve from CAPM → FF3 → FF5? What does that tell us?
- What do the SMB, HML, RMW, CMA loadings say about each stock's characteristics?
- Does alpha persist at the portfolio level after factor adjustment, or does it disappear?

# 6. Integration & Investment Insight

*[TO BE COMPLETED AFTER TUESDAY DISCUSSION]*

This section synthesizes findings across all analytical components.

### Key Questions to Address
- What does the combined analysis (volatility, portfolio risk, factor exposures) tell us about this four-stock portfolio?
- Which portfolio construction approach best serves the stated investment objective?
- Where does the return come from — factor exposures or idiosyncratic behavior?
- What are the limitations of the analysis (short time period, data frequency, model assumptions)?
- What recommendation would we make to an investor considering these stocks?

### Connecting to Course Themes
The AI-versus-humans framing from Chapter 1 of the textbook applies here: the factor regressions and portfolio optimizers provide the quantitative baseline, but interpretation and judgment are where investment value is created. Factor models tell us where returns came from; they do not tell us which risks we should *want* to take.

In [ ]:
# Summary dashboard — everything on one page

print("=" * 80)
print("CAPSTONE PROJECT SUMMARY DASHBOARD")
print("=" * 80)

print(f"\nAnalysis period: {common_dates.min().date()} to {common_dates.max().date()}")
print(f"Trading days: {len(common_dates)}")
print(f"Risk-free rate (annualized): {rf_annual*100:.2f}%")

print("\n1. STOCK-LEVEL SUMMARY")
print("-" * 80)
print(f"{'Ticker':<8} {'Ann Ret':>10} {'Ann Vol':>10} {'Sharpe':>10} {'CAPM Beta':>12} {'FF5 R²':>10}")
print("-" * 80)
for t in stock_tickers:
    ann_ret = returns[t].mean() * 252 * 100
    ann_vol = returns[t].std() * np.sqrt(252) * 100
    sharpe = (returns[t].mean() * 252 - rf_annual) / (returns[t].std() * np.sqrt(252))
    beta = capm_results[t].params['Mkt-RF']
    ff5_r2 = ff5_results[t].rsquared
    print(f"{t:<8} {ann_ret:>9.2f}% {ann_vol:>9.2f}% {sharpe:>10.4f} {beta:>12.3f} {ff5_r2:>10.4f}")

print("\n2. PORTFOLIO-LEVEL SUMMARY")
print("-" * 80)
print(f"{'Portfolio':<18} {'Ann Ret':>10} {'Ann Vol':>10} {'Sharpe':>10}")
print("-" * 80)
for name, r, v, s in [('Equal-Weight', ew_ret, ew_vol, ew_sharpe),
                       ('Min-Variance', mv_ret, mv_vol, mv_sharpe),
                       ('Max Sharpe', ms_ret, ms_vol, ms_sharpe)]:
    print(f"{name:<18} {r*100:>9.2f}% {v*100:>9.2f}% {s:>10.4f}")

print("\n3. RISK MEASURES (99% Historical)")
print("-" * 80)
print(f"{'Portfolio':<18} {'Daily VaR':>12} {'Daily CVaR':>12}")
print("-" * 80)
for name, rets in port_rets.items():
    var99 = np.percentile(rets, 1)
    cvar99 = rets[rets <= var99].mean()
    print(f"{name:<18} {var99*100:>11.3f}% {cvar99*100:>11.3f}%")

print("\n" + "=" * 80)
print("End of analysis.")
print("=" * 80)